# misen + LangGraph Integration

How to use misen blocks inside LangGraph nodes.

Key insight: misen blocks are `dict → dict`, and LangGraph nodes are `State → dict` — they connect naturally.

## Setup

Before running this notebook, install dependencies from the **repo root**:

```bash
cd misen
uv pip install -e ".[dev]" langgraph
```

Then open this notebook with the `.venv` kernel.

In [ ]:
# Verify setup
import misen
import langgraph
print(f"misen {misen.__version__}, langgraph OK")

## 1. Define misen blocks

Define reusable blocks first. These work independently — no LangGraph dependency required.

In [ ]:
from misen import tool, sequential, parallel, branch


@tool
def parse_document(input: dict) -> dict:
    """Parse a document and extract text."""
    file_path = input["file_path"]
    return {
        "text": f"Parsed content from {file_path}",
        "doc_type": "report" if "report" in file_path else "memo",
    }


@tool
def chunk_text(input: dict) -> dict:
    """Split text into chunks."""
    text = input["text"]
    chunks = [text[i:i+100] for i in range(0, len(text), 80)]
    return {"chunks": chunks}


@tool
def embed_chunks(input: dict) -> dict:
    """Convert chunks to embeddings."""
    chunks = input["chunks"]
    vectors = [[0.1 * i] * 3 for i, _ in enumerate(chunks)]
    return {"vectors": vectors}


@tool
def extract_keywords(input: dict) -> dict:
    """Extract keywords from text."""
    text = input["text"]
    keywords = text.split()[:5]
    return {"keywords": keywords}


@tool
def generate_summary(input: dict) -> dict:
    """Generate a summary."""
    text = input["text"]
    return {"summary": f"Summary of: {text[:50]}..."}

## 2. Run with misen only (no LangGraph)

Compose blocks into pipelines and run them directly.

In [ ]:
# Ingest pipeline: parse → chunk → embed
ingest = parse_document | chunk_text | embed_chunks

# Analysis pipeline: keywords + summary in parallel
analyze = parallel(extract_keywords, generate_summary)

# Full pipeline
full_pipeline = sequential(ingest, analyze)

# Run
result = full_pipeline.run_sync({"file_path": "report_2024.hwp"})

print("chunks:", len(result["chunks"]))
print("vectors:", len(result["vectors"]))
print("keywords:", result["keywords"])
print("summary:", result["summary"])
print("doc_type:", result["doc_type"])

## 3. Use misen blocks in LangGraph

Call misen blocks inside LangGraph node functions.

### Approach A: Call blocks directly inside node functions

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END


# LangGraph State definition
class DocState(TypedDict, total=False):
    file_path: str
    text: str
    doc_type: str
    chunks: list[str]
    vectors: list[list[float]]
    keywords: list[str]
    summary: str
    decision: str
    analysis_type: str
    result: str


# Call misen blocks inside node functions
async def ingest_node(state: DocState) -> dict:
    """Use misen's ingest pipeline as a LangGraph node."""
    return await ingest.run(dict(state))


async def analyze_node(state: DocState) -> dict:
    """Use misen's analyze pipeline as a LangGraph node."""
    return await analyze.run(dict(state))


async def decide_node(state: DocState) -> dict:
    """Route based on document type."""
    if state["doc_type"] == "report":
        return {"decision": "detailed_review"}
    return {"decision": "quick_scan"}

In [ ]:
# Build the graph
graph = StateGraph(DocState)

graph.add_node("ingest", ingest_node)
graph.add_node("analyze", analyze_node)
graph.add_node("decide", decide_node)

graph.add_edge(START, "ingest")
graph.add_edge("ingest", "analyze")
graph.add_edge("analyze", "decide")
graph.add_edge("decide", END)

app = graph.compile()

# Run
result = await app.ainvoke({"file_path": "report_2024.hwp"})
print(result)

### Approach B: Helper function for cleaner conversion

Extract the repeated `dict(state)` conversion into a helper.

In [ ]:
from misen import Block


def as_node(block: Block):
    """Convert a misen Block to a LangGraph node function."""
    async def node(state: dict) -> dict:
        return await block.run(dict(state))
    node.__name__ = block.name
    return node


# Usage
graph2 = StateGraph(DocState)

graph2.add_node("ingest", as_node(ingest))
graph2.add_node("analyze", as_node(analyze))
graph2.add_node("decide", decide_node)

graph2.add_edge(START, "ingest")
graph2.add_edge("ingest", "analyze")
graph2.add_edge("analyze", "decide")
graph2.add_edge("decide", END)

app2 = graph2.compile()
result = await app2.ainvoke({"file_path": "memo_draft.txt"})
print(result)

## 4. Why use misen with LangGraph?

### Can't I just use LangGraph alone?

You can. But adding misen gives you:

1. **Reuse the same blocks across projects** — the `ingest` pipeline works in Project A's LangGraph graph, Project B's script, or a FastAPI endpoint
2. **No LangGraph dependency for testing** — run blocks standalone in tests, scripts, or notebooks
3. **Complex composition inside a single node** — use `sequential`, `parallel`, `branch` freely within one LangGraph node

```python
# A complex misen pipeline running as ONE LangGraph node
complex_pipeline = sequential(
    parse_document,
    branch(
        lambda d: d["doc_type"] == "report",
        sequential(chunk_text, embed_chunks),  # report: chunk + embed
        extract_keywords,                       # memo: keywords only
    ),
    generate_summary,
)
```

## 5. Conditional routing with LangGraph + misen

Using misen's `branch` alongside LangGraph's graph structure.

In [ ]:
@tool
def detailed_analysis(input: dict) -> dict:
    """Detailed analysis: chunking + embedding + keywords + summary."""
    return {"analysis_type": "detailed", "result": f"Detailed analysis of {input['text'][:30]}"}


@tool
def quick_analysis(input: dict) -> dict:
    """Quick analysis: keywords only."""
    return {"analysis_type": "quick", "result": f"Quick scan of {input['text'][:30]}"}


# Define branching logic with misen
smart_analysis = branch(
    lambda d: d.get("doc_type") == "report",
    detailed_analysis,
    quick_analysis,
)

# Use in LangGraph — the entire misen branch becomes one node
graph3 = StateGraph(DocState)

graph3.add_node("parse", as_node(parse_document))
graph3.add_node("smart_analysis", as_node(smart_analysis))

graph3.add_edge(START, "parse")
graph3.add_edge("parse", "smart_analysis")
graph3.add_edge("smart_analysis", END)

app3 = graph3.compile()

# report → detailed
r1 = await app3.ainvoke({"file_path": "report_2024.hwp"})
print("report:", r1["analysis_type"])  # detailed

# memo → quick
r2 = await app3.ainvoke({"file_path": "memo_draft.txt"})
print("memo:", r2["analysis_type"])  # quick

## Summary

| Pattern | Code |
|---|---|
| Block as node | `graph.add_node("name", as_node(block))` |
| Pipeline as node | `graph.add_node("name", as_node(parse \| chunk \| embed))` |
| Branch as node | `graph.add_node("name", as_node(branch(cond, a, b)))` |
| Standalone run | `result = block.run_sync({...})` |

`as_node` is a 3-line helper. It's not an adapter layer — just a function call.